In [1]:
!git clone https://github.com/Project-DiffShield/DiffShield.git
import sys
sys.path.append('/kaggle/working/DiffShield')

!pip install -q kornia diffusers transformers optuna lpips accelerate scikit-learn kneed matplotlib pandas
print("Environment dependencies initialized.")

Cloning into 'DiffShield'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 3), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 424.02 KiB | 4.12 MiB/s, done.
Resolving deltas: 100% (3/3), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.6 MB/s eta 0:00:00
Environment dependencies initialized.


In [2]:
import torch
import numpy as np
import math
import os
import shutil
import zipfile
import json
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as T
from PIL import Image
import lpips
from torchvision.utils import save_image, make_grid

from src.losses import DiffShieldLoss
from src.optimizer import PGDOptimizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Executing on device: {device}")

ARTIFACTS_DIR = '/kaggle/working/artifacts'
METRICS_DIR = '/kaggle/working/metrics'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

loss_fn = DiffShieldLoss(device=device)
lpips_vgg = lpips.LPIPS(net='vgg').to(device)
target_concept_embedding = loss_fn.encode_target_text(["a potted plant"])

def compute_image_quality_metrics(clean_tensor, immunized_tensor):
    clean_np = ((clean_tensor.squeeze(0).cpu().numpy() + 1.0) * 127.5).astype(np.uint8)
    immunized_np = ((immunized_tensor.squeeze(0).cpu().numpy() + 1.0) * 127.5).astype(np.uint8)
    
    mse = np.mean((clean_np.astype(np.float64) - immunized_np.astype(np.float64)) ** 2)
    psnr = 20 * math.log10(255.0 / math.sqrt(mse)) if mse > 0 else float('inf')
    
    C1 = (0.01 * 255) ** 2
    C2 = (0.03 * 255) ** 2
    mu1, mu2 = clean_np.mean(), immunized_np.mean()
    s1_sq, s2_sq = clean_np.var(), immunized_np.var()
    s12 = ((clean_np - mu1) * (immunized_np - mu2)).mean()
    ssim = ((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (s1_sq + s2_sq + C2))
    
    with torch.no_grad():
        lpips_score = lpips_vgg(clean_tensor, immunized_tensor).item()
        
    linf = (immunized_tensor - clean_tensor).abs().max().item()
    return {"MSE": mse, "PSNR": psnr, "SSIM": ssim, "LPIPS": lpips_score, "Linf": linf}

print("Backbones, directories, and metric computation functions initialized.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Executing on device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

...

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 232MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
Backbones, directories, and metric computation functions initialized.


In [3]:
dataset_base = None
img_dir = None
attr_file_path = None

# Scan /kaggle/input for the specific CelebAMask-HQ folder structure
for root, dirs, files in os.walk('/kaggle/input'):
    if 'CelebA-HQ-img' in dirs and 'CelebAMask-HQ-attribute-anno.txt' in files:
        dataset_base = root
        img_dir = os.path.join(root, 'CelebA-HQ-img')
        attr_file_path = os.path.join(root, 'CelebAMask-HQ-attribute-anno.txt')
        break

if not img_dir or not attr_file_path:
    raise FileNotFoundError("Could not locate CelebA-HQ-img or CelebAMask-HQ-attribute-anno.txt. Check dataset attachment.")

print(f"Dataset mapped. Images: {img_dir}")
print(f"Attributes mapped: {attr_file_path}")

with open(attr_file_path, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

num_images = int(lines[0])
attr_names = lines[1].split()
data = []
img_filenames = []

for line in lines[2:]:
    parts = line.split()
    img_filenames.append(parts[0])
    # Map raw 1 and -1 to binary 1 and 0 for clustering
    data.append([1 if int(x) == 1 else 0 for x in parts[1:]])

attr_matrix = np.array(data, dtype=np.float32)
print(f"Loaded {attr_matrix.shape[0]} images across {attr_matrix.shape[1]} binary attributes.")

Dataset mapped. Images: /kaggle/input/datasets/ipythonx/celebamaskhq/CelebAMask-HQ/CelebA-HQ-img
Attributes mapped: /kaggle/input/datasets/ipythonx/celebamaskhq/CelebAMask-HQ/CelebAMask-HQ-attribute-anno.txt
Loaded 30000 images across 40 binary attributes.


In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from kneed import KneeLocator

wcss = []
k_range = list(range(1, 21))

print("Computing WCSS across candidate cluster ranges (1 to 20)...")
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(attr_matrix)
    wcss.append(km.inertia_)

wcss_df = pd.DataFrame({"K": k_range, "WCSS": wcss})
wcss_df.to_csv(os.path.join(METRICS_DIR, 'wcss_elbow_values.csv'), index=False)

kl = KneeLocator(k_range, wcss, curve="convex", direction="decreasing")
k_calib = int(kl.elbow) if kl.elbow is not None else 8
print(f"Mathematical Elbow Detected at K = {k_calib}")

plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', color='#A4123F', linewidth=2, markersize=6)
plt.axvline(x=k_calib, color='navy', linestyle='--', label=f'Optimal K ({k_calib})')
plt.title('Elbow Method: Attribute Variance Clustering', fontsize=12, fontweight='bold')
plt.xlabel('Number of Clusters (K)', fontsize=11)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
elbow_plot_path = os.path.join(ARTIFACTS_DIR, 'elbow_method_wcss_curve.png')
plt.savefig(elbow_plot_path, dpi=300, bbox_inches='tight')
plt.close()

kmeans_calib = KMeans(n_clusters=k_calib, init='k-means++', random_state=42, n_init=10)
kmeans_calib.fit(attr_matrix)
centroid_indices_calib, _ = pairwise_distances_argmin_min(kmeans_calib.cluster_centers_, attr_matrix)
calib_filenames = [img_filenames[idx] for idx in centroid_indices_calib]

calib_manifest_df = pd.DataFrame({
    "Calibration_Cluster_ID": list(range(1, k_calib + 1)),
    "Original_Index": centroid_indices_calib,
    "Filename": calib_filenames
})
calib_manifest_df.to_csv(os.path.join(METRICS_DIR, 'calibration_subset_manifest.csv'), index=False)

calib_dir = '/kaggle/working/calibration_subset'
os.makedirs(calib_dir, exist_ok=True)

# Downscale from 1024x1024 to 512x512 during extraction
for fname in calib_filenames:
    src_path = os.path.join(img_dir, fname)
    if os.path.exists(src_path):
        img = Image.open(src_path).convert('RGB')
        img_resized = img.resize((512, 512), Image.Resampling.BICUBIC)
        img_resized.save(os.path.join(calib_dir, fname))

print(f"Calibration Manifest saved. Downscaled and stored {len(os.listdir(calib_dir))} images in {calib_dir}")

Computing WCSS across candidate cluster ranges (1 to 20)...
Mathematical Elbow Detected at K = 4
Calibration Manifest saved. Downscaled and stored 4 images in /kaggle/working/calibration_subset


In [5]:
calib_set = set(centroid_indices_calib)
eval_indices_available = [i for i in range(len(img_filenames)) if i not in calib_set]

eval_attr_matrix = attr_matrix[eval_indices_available]
eval_filenames_available = [img_filenames[i] for i in eval_indices_available]

K_EVAL = 70
print(f"Clustering {eval_attr_matrix.shape[0]} disjoint images into {K_EVAL} attribute centroids...")
kmeans_eval = KMeans(n_clusters=K_EVAL, init='k-means++', random_state=42, n_init=10)
kmeans_eval.fit(eval_attr_matrix)

centroid_indices_eval, _ = pairwise_distances_argmin_min(kmeans_eval.cluster_centers_, eval_attr_matrix)
eval_final_filenames = [eval_filenames_available[idx] for idx in centroid_indices_eval]

eval_manifest_df = pd.DataFrame({
    "Eval_Image_ID": [f"face_{i+1:03d}" for i in range(K_EVAL)],
    "Original_Index": [eval_indices_available[idx] for idx in centroid_indices_eval],
    "Filename": eval_final_filenames
})
eval_manifest_df.to_csv(os.path.join(METRICS_DIR, 'evaluation_70_manifest.csv'), index=False)

eval_dir = '/kaggle/working/diverse_70_images'
os.makedirs(eval_dir, exist_ok=True)

for fname in eval_final_filenames:
    src_path = os.path.join(img_dir, fname)
    if os.path.exists(src_path):
        img = Image.open(src_path).convert('RGB')
        img_resized = img.resize((512, 512), Image.Resampling.BICUBIC)
        img_resized.save(os.path.join(eval_dir, fname))

print(f"Data isolation complete. Downscaled 70 evaluation centroids to 512x512 in {eval_dir}")

Clustering 29996 disjoint images into 70 attribute centroids...
Data isolation complete. Downscaled 70 evaluation centroids to 512x512 in /kaggle/working/diverse_70_images


In [6]:
from torch.utils.data import DataLoader
from src.data import get_dataloader

calib_loader = get_dataloader(root_dir=calib_dir, batch_size=int(k_calib), image_size=512)
calib_batch, _ = next(iter(calib_loader))
calib_batch = calib_batch.to(device)

epsilon = 8 / 255
delta = torch.zeros_like(calib_batch).to(device)
delta.uniform_(-epsilon, epsilon)
poisoned_batch = torch.clamp(calib_batch + delta, -1.0, 1.0)

raw_vis = loss_fn.compute_visual_loss(calib_batch, poisoned_batch).item()
raw_sem = loss_fn.compute_semantic_loss(poisoned_batch, target_concept_embedding).item()
raw_str = loss_fn.compute_structure_loss(calib_batch, poisoned_batch).item()

alpha_base = 1.0 / max(raw_vis, 1e-4)
beta_base  = 1.0 / max(raw_sem, 1e-4)
gamma_base = 1.0 / max(raw_str, 1e-4)

base_config = {
    "raw_losses": {"visual": raw_vis, "semantic": raw_sem, "structural": raw_str},
    "base_multipliers": {"alpha_base": alpha_base, "beta_base": beta_base, "gamma_base": gamma_base}
}
with open(os.path.join(METRICS_DIR, 'base_multipliers.json'), 'w') as f:
    json.dump(base_config, f, indent=4)

print(f"Base multipliers computed and saved to {METRICS_DIR}/base_multipliers.json")

Base multipliers computed and saved to /kaggle/working/metrics/base_multipliers.json


In [7]:
multipliers = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]
sweep_logs = []

def sweep_subset_bounds(param_name):
    valid_mults = []
    print(f"\n--- Sweeping Search Boundaries for {param_name} ---")
    
    for mult in multipliers:
        w_a = alpha_base * (mult if param_name == 'alpha' else 1.0)
        w_b = beta_base  * (mult if param_name == 'beta'  else 1.0)
        w_g = gamma_base * (mult if param_name == 'gamma' else 1.0)
        
        optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device)
        batch_passed = True
        min_psnr_batch = float('inf')
        min_ssim_batch = float('inf')
        
        for i in range(calib_batch.size(0)):
            img = calib_batch[i:i+1]
            immunized = optimizer.optimize(img, target_concept_embedding, w_alpha=w_a, w_beta=w_b, w_gamma=w_g)
            m = compute_image_quality_metrics(img, immunized)
            min_psnr_batch = min(min_psnr_batch, m['PSNR'])
            min_ssim_batch = min(min_ssim_batch, m['SSIM'])
            if m['PSNR'] < 38.0 or m['SSIM'] < 0.95:
                batch_passed = False
                
        status = "PASS" if batch_passed else "FAIL"
        sweep_logs.append({
            "parameter": param_name, "multiplier": mult,
            "min_batch_psnr": min_psnr_batch, "min_batch_ssim": min_ssim_batch,
            "status": status
        })
        print(f"Multiplier {mult:4.2f}x | Min PSNR: {min_psnr_batch:.2f} dB | Min SSIM: {min_ssim_batch:.4f} | Status: {status}")
        if batch_passed:
            valid_mults.append(mult)
            
    min_m = min(valid_mults) if valid_mults else 0.1
    max_m = max(valid_mults) if valid_mults else 1.0
    return min_m, max_m

min_a, max_a = sweep_subset_bounds('alpha')
min_b, max_b = sweep_subset_bounds('beta')
min_g, max_g = sweep_subset_bounds('gamma')

min_alpha, max_alpha = alpha_base * min_a, alpha_base * max_a
min_beta,  max_beta  = beta_base  * min_b, beta_base  * max_b
min_gamma, max_gamma = gamma_base * min_g, gamma_base * max_g

pd.DataFrame(sweep_logs).to_csv(os.path.join(METRICS_DIR, 'boundary_sweeps_log.csv'), index=False)

bounds_dict = {
    "alpha_bounds": [min_alpha, max_alpha],
    "beta_bounds": [min_beta, max_beta],
    "gamma_bounds": [min_gamma, max_gamma]
}
with open(os.path.join(METRICS_DIR, 'optuna_search_boundaries.json'), 'w') as f:
    json.dump(bounds_dict, f, indent=4)

print(f"Sweep logs and search boundaries persisted to {METRICS_DIR}/")


--- Sweeping Search Boundaries for alpha ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.la

Iter   0: Total +120.0340 | L_vis 18.0827 | L_sem 0.8043 (cos_sim=0.1957) | L_str 0.0253
Iter  10: Total +130.5408 | L_vis 21.5106 | L_sem 0.7905 (cos_sim=0.2095) | L_str 0.0273
Iter  20: Total +18.7008 | L_vis 8.3888 | L_sem 0.7770 (cos_sim=0.2230) | L_str 0.0034
Iter  30: Total +140.6278 | L_vis 22.4577 | L_sem 0.7966 (cos_sim=0.2034) | L_str 0.0294

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +4.6567 | L_vis 1.6422 | L_sem 0.7958 (cos_sim=0.2042) | L_str 0.0011
Iter  10: Total +16.0787 | L_vis 9.0758 | L_sem 0.7564 (cos_sim=0.2436) | L_str 0.0028
Iter  20: Total +21.8855 | L_vis 12.0490 | L_sem 0.7524 (cos_sim=0.2476) | L_str 0.0037
Iter  30: Total +21.7070 | L_vis 9.4275 | L_sem 0.7983 (cos_sim=0.2017) | L_str 0.0040

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +141.0705 | L_vis 41.7465 | L_sem 0.8148 (cos_sim=0.1852) | L_str 0.0272
Iter  10: Total +138.1229 | L_vis 23.6752 | L_sem 0.8155 (cos_sim=0.1845) | L_str 0.0287
It

In [8]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    w_a = trial.suggest_float('alpha', min_alpha, max_alpha)
    w_b = trial.suggest_float('beta',  min_beta,  max_beta)
    w_g = trial.suggest_float('gamma', min_gamma, max_gamma)
    
    optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device)
    subset_losses = []
    
    for i in range(calib_batch.size(0)):
        img = calib_batch[i:i+1]
        immunized = optimizer.optimize(img, target_concept_embedding, w_alpha=w_a, w_beta=w_b, w_gamma=w_g)
        m = compute_image_quality_metrics(img, immunized)
        
        if m['PSNR'] < 38.0 or m['SSIM'] < 0.95:
            return -9999.0
            
        total_loss, _, _, _ = loss_fn(img, immunized, target_concept_embedding, alpha=w_a, beta=w_b, gamma=w_g)
        subset_losses.append(total_loss.item())
        
    return sum(subset_losses) / len(subset_losses)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

opt_alpha = study.best_params['alpha']
opt_beta  = study.best_params['beta']
opt_gamma = study.best_params['gamma']

trials_df = study.trials_dataframe()
trials_df.to_csv(os.path.join(METRICS_DIR, 'optuna_all_trials_log.csv'), index=False)

best_params_record = {
    "best_trial_number": study.best_trial.number,
    "best_objective_value": study.best_value,
    "optimal_weights": {
        "alpha": opt_alpha,
        "beta": opt_beta,
        "gamma": opt_gamma
    }
}
with open(os.path.join(METRICS_DIR, 'optimal_hyperparameters.json'), 'w') as f:
    json.dump(best_params_record, f, indent=4)

valid_scores = [t.value for t in study.trials if t.value is not None and t.value > -9000]
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(valid_scores) + 1), valid_scores, marker='s', color='darkgreen', linewidth=2)
plt.title('Optuna Bayesian Optimization: Objective Progression', fontsize=12, fontweight='bold')
plt.xlabel('Valid Trial Count', fontsize=11)
plt.ylabel('Mean Adversarial Calibration Loss', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
optuna_plot_path = os.path.join(ARTIFACTS_DIR, 'optuna_convergence_history.png')
plt.savefig(optuna_plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\nFinal Hyperparameters Saved: Alpha={opt_alpha:.4f}, Beta={opt_beta:.4f}, Gamma={opt_gamma:.4f}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.la

Iter   0: Total +44.8536 | L_vis 14.8412 | L_sem 0.8132 (cos_sim=0.1868) | L_str 0.0241
Iter  10: Total +13.4530 | L_vis 7.0402 | L_sem 0.8138 (cos_sim=0.1862) | L_str 0.0037
Iter  20: Total +62.7064 | L_vis 22.4354 | L_sem 0.7838 (cos_sim=0.2162) | L_str 0.0250
Iter  30: Total +29.0137 | L_vis 14.1719 | L_sem 0.7772 (cos_sim=0.2228) | L_str 0.0027

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total -1.2162 | L_vis 1.1920 | L_sem 0.8029 (cos_sim=0.1971) | L_str 0.0014
Iter  10: Total +53.7183 | L_vis 16.2652 | L_sem 0.7890 (cos_sim=0.2110) | L_str 0.0323
Iter  20: Total +25.0391 | L_vis 12.4543 | L_sem 0.7597 (cos_sim=0.2403) | L_str 0.0023
Iter  30: Total +32.8959 | L_vis 15.9071 | L_sem 0.7424 (cos_sim=0.2576) | L_str 0.0023

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +95.9440 | L_vis 36.0842 | L_sem 0.8151 (cos_sim=0.1849) | L_str 0.0289
Iter  10: Total +102.4009 | L_vis 39.0633 | L_sem 0.8153 (cos_sim=0.1847) | L_str 0.0285
Iter

...

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.la

Iter   0: Total +69.7362 | L_vis 21.0181 | L_sem 0.8063 (cos_sim=0.1937) | L_str 0.0245
Iter  10: Total +17.3628 | L_vis 5.9792 | L_sem 0.8221 (cos_sim=0.1779) | L_str 0.0025
Iter  20: Total +68.9470 | L_vis 20.6324 | L_sem 0.8173 (cos_sim=0.1827) | L_str 0.0250
Iter  30: Total +31.3628 | L_vis 11.0842 | L_sem 0.8221 (cos_sim=0.1779) | L_str 0.0030

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +5.5140 | L_vis 1.8501 | L_sem 0.7968 (cos_sim=0.2032) | L_str 0.0012
Iter  10: Total +16.9851 | L_vis 5.9565 | L_sem 0.8061 (cos_sim=0.1939) | L_str 0.0020
Iter  20: Total +78.6512 | L_vis 22.8974 | L_sem 0.8135 (cos_sim=0.1865) | L_str 0.0317
Iter  30: Total +73.3937 | L_vis 20.6735 | L_sem 0.8032 (cos_sim=0.1968) | L_str 0.0330

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +116.8041 | L_vis 37.7993 | L_sem 0.8285 (cos_sim=0.1715) | L_str 0.0278
Iter  10: Total +20.6195 | L_vis 7.2662 | L_sem 0.8125 (cos_sim=0.1875) | L_str 0.0021
Iter  

In [9]:
eval_loader = get_dataloader(root_dir=eval_dir, batch_size=1, image_size=512)
output_protected_dir = '/kaggle/working/stage1_protected_images'
os.makedirs(output_protected_dir, exist_ok=True)

detailed_metrics = []
optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device)
sample_visuals = []

print(f"Processing 70 Disjoint Centroids using Optimal Hyperparameters...")
for idx, (clean_img, path) in enumerate(eval_loader):
    clean_img = clean_img.to(device)
    raw_fname = os.path.basename(path[0])
    
    immunized_img = optimizer.optimize(
        clean_img, target_concept_embedding,
        w_alpha=opt_alpha, w_beta=opt_beta, w_gamma=opt_gamma
    )
    
    m = compute_image_quality_metrics(clean_img, immunized_img)
    
    protected_norm = (immunized_img.squeeze(0) + 1.0) / 2.0
    clean_norm = (clean_img.squeeze(0) + 1.0) / 2.0
    
    save_filename = f"protected_face_{idx+1:03d}.png"
    save_image(protected_norm, os.path.join(output_protected_dir, save_filename))
    
    detailed_metrics.append({
        "Image_ID": f"face_{idx+1:03d}",
        "Original_Filename": raw_fname,
        "Protected_Filename": save_filename,
        "PSNR_dB": m['PSNR'],
        "SSIM": m['SSIM'],
        "LPIPS": m['LPIPS'],
        "Linf": m['Linf'],
        "MSE": m['MSE'],
        "PSNR_Passed": m['PSNR'] >= 38.0,
        "SSIM_Passed": m['SSIM'] >= 0.95
    })
    
    if idx < 4:
        sample_visuals.extend([clean_norm.cpu(), protected_norm.cpu()])
        
    if (idx + 1) % 10 == 0 or (idx + 1) == 70:
        print(f"[{idx+1:02d}/70] -> PSNR: {m['PSNR']:.2f} dB | SSIM: {m['SSIM']:.4f} | LPIPS: {m['LPIPS']:.4f}")

per_image_df = pd.DataFrame(detailed_metrics)
per_image_df.to_csv(os.path.join(METRICS_DIR, 'stage1_per_image_metrics.csv'), index=False)

grid = make_grid(sample_visuals, nrow=2, padding=10, normalize=False)
grid_np = grid.permute(1, 2, 0).numpy()

plt.figure(figsize=(8, 8))
plt.imshow(grid_np)
plt.axis('off')
plt.title('DiffShield Phase 1: Clean (Left) vs. Protected (Right)', fontsize=12, fontweight='bold')
comp_plot_path = os.path.join(ARTIFACTS_DIR, 'phase1_sample_comparisons.png')
plt.savefig(comp_plot_path, dpi=300, bbox_inches='tight')
plt.close()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_att

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.la

Processing 70 Disjoint Centroids using Optimal Hyperparameters...
Iter   0: Total +3.7765 | L_vis 1.2962 | L_sem 0.8073 (cos_sim=0.1927) | L_str 0.0011
Iter  10: Total +59.3669 | L_vis 18.2631 | L_sem 0.8084 (cos_sim=0.1916) | L_str 0.0250
Iter  20: Total +21.4558 | L_vis 7.7895 | L_sem 0.8121 (cos_sim=0.1879) | L_str 0.0021
Iter  30: Total +84.7328 | L_vis 27.7408 | L_sem 0.7930 (cos_sim=0.2070) | L_str 0.0254

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +60.2013 | L_vis 18.8054 | L_sem 0.8305 (cos_sim=0.1695) | L_str 0.0236
Iter  10: Total +13.8746 | L_vis 5.0158 | L_sem 0.8350 (cos_sim=0.1650) | L_str 0.0016
Iter  20: Total +18.6357 | L_vis 6.7504 | L_sem 0.8383 (cos_sim=0.1617) | L_str 0.0020
Iter  30: Total +80.2580 | L_vis 25.9345 | L_sem 0.8178 (cos_sim=0.1822) | L_str 0.0262

Final L∞ norm of perturbation: 0.031373 (limit: 0.031373)
Iter   0: Total +6.0103 | L_vis 2.1196 | L_sem 0.7965 (cos_sim=0.2035) | L_str 0.0012
Iter  10: Total +19.5626 | L_vi

In [10]:
psnr_vals = [r['PSNR_dB'] for r in detailed_metrics]
ssim_vals = [r['SSIM'] for r in detailed_metrics]
lpips_vals = [r['LPIPS'] for r in detailed_metrics]
linf_vals = [r['Linf'] for r in detailed_metrics]

summary_records = [
    {
        "Metric": "PSNR (dB)",
        "Mean": np.mean(psnr_vals), "Std": np.std(psnr_vals),
        "Min": np.min(psnr_vals), "Max": np.max(psnr_vals),
        "Threshold": ">= 38.0",
        "Violations": sum(1 for p in psnr_vals if p < 38.0)
    },
    {
        "Metric": "SSIM",
        "Mean": np.mean(ssim_vals), "Std": np.std(ssim_vals),
        "Min": np.min(ssim_vals), "Max": np.max(ssim_vals),
        "Threshold": ">= 0.95",
        "Violations": sum(1 for s in ssim_vals if s < 0.95)
    },
    {
        "Metric": "LPIPS",
        "Mean": np.mean(lpips_vals), "Std": np.std(lpips_vals),
        "Min": np.min(lpips_vals), "Max": np.max(lpips_vals),
        "Threshold": "< 0.05",
        "Violations": sum(1 for l in lpips_vals if l > 0.05)
    },
    {
        "Metric": "Linf",
        "Mean": np.mean(linf_vals), "Std": np.std(linf_vals),
        "Min": np.min(linf_vals), "Max": np.max(linf_vals),
        "Threshold": "<= 0.0314",
        "Violations": sum(1 for li in linf_vals if li > (8/255 + 1e-4))
    }
]

summary_df = pd.DataFrame(summary_records)
summary_df.to_csv(os.path.join(METRICS_DIR, 'stage1_summary_metrics.csv'), index=False)
with open(os.path.join(METRICS_DIR, 'stage1_summary_metrics.json'), 'w') as f:
    json.dump(summary_records, f, indent=4)

print("\n" + "=" * 80)
print(f"  STAGE 1 GENERALIZATION EVALUATION SUMMARY (70 Disjoint Images)")
print("=" * 80)
print(f"{'Metric':<12} | {'Mean ± Std':<18} | {'Min':<10} | {'Max':<10} | {'Threshold':<12} | {'Violations'}")
print("-" * 80)
for r in summary_records:
    print(f"{r['Metric']:<12} | {r['Mean']:6.4f} ± {r['Std']:5.4f}    | {r['Min']:8.4f}   | {r['Max']:8.4f}   | {r['Threshold']:<12} | {r['Violations']}/70")
print("=" * 80)

print("\nCreating downloadable bundles...")
!zip -r -q /kaggle/working/diffshield_phase1_complete_results.zip /kaggle/working/metrics /kaggle/working/artifacts
!zip -r -q /kaggle/working/stage1_protected_images.zip /kaggle/working/stage1_protected_images
!zip -r -q /kaggle/working/stage1_original_images.zip /kaggle/working/diverse_70_images


  STAGE 1 GENERALIZATION EVALUATION SUMMARY (70 Disjoint Images)
Metric       | Mean ± Std         | Min        | Max        | Threshold    | Violations
--------------------------------------------------------------------------------
PSNR (dB)    | 38.8610 ± 0.2042    |  38.4965   |  39.4531   | >= 38.0      | 0/70
SSIM         | 0.9990 ± 0.0003    |   0.9978   |   0.9996   | >= 0.95      | 0/70
LPIPS        | 0.2772 ± 0.0598    |   0.1658   |   0.4212   | < 0.05       | 70/70
Linf         | 0.0314 ± 0.0000    |   0.0314   |   0.0314   | <= 0.0314    | 0/70

Creating downloadable bundles...
